# Use Intended Prompt for Best Performance

This notebook loads the released Shaer adapter, uses the intended paper prompt format, and pulls real prompt examples from the released split dataset's `test` split.

Released artifacts:

- Model: `Shaer-AI/Shaer-adapters`
- Base model: `Navid-AI/Yehia-7B-preview`
- Split dataset: `Shaer-AI/ashaar-with-enhanced-descriptions-baseform-final-sft-lte20-min500-splits`

In [ ]:
!pip -q install transformers peft accelerate sentencepiece datasets

In [ ]:
import random
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

In [ ]:
DATASET_REPO = "Shaer-AI/ashaar-with-enhanced-descriptions-baseform-final-sft-lte20-min500-splits"
BASE_MODEL = "Navid-AI/Yehia-7B-preview"
ADAPTER_REPO = "Shaer-AI/Shaer-adapters"

test_rows = load_dataset(DATASET_REPO, split="test")
print(test_rows)

In [ ]:
def show_example(row):
    return {
        "base_meter": row["base_meter"],
        "form": row["form"],
        "requested_num_lines": row.get("requested_num_lines", row.get("sft_num_lines")),
        "enhanced_description": row["enhanced_description"],
    }

for idx in [0, 1, 2]:
    print(f"example_{idx}", show_example(test_rows[idx]))

In [ ]:
SYSTEM_PROMPT = """أنت شاعر عربي تكتب الشعر العمودي الكلاسيكي.
التزم بالبحر المحدد في كل شطر، واستلهم من الموضوع دون نقله حرفياً.
أخرج الأبيات فقط دون مقدمة أو تعليق.
التزم التزاماً صارماً بالبحر المطلوب، ولا تخرج عنه."""

def build_prompt(base_meter: str, form: str, num_lines: int, description: str) -> str:
    user_prompt = f"""البحر الأساسي: {base_meter}
الصيغة: {form}
عدد الأشطر المطلوب: {num_lines}
الموضوع: {description}

اكتب {num_lines} أشطارًا ملتزمة بصيغة {form} من بحر {base_meter} دون أي شرح إضافي."""
    return f"<s> [INST] <<SYS>>\n{SYSTEM_PROMPT}\n<</SYS>>\n\n{user_prompt.strip()} [/INST]"

selected_index = 0
row = test_rows[selected_index]
prompt = build_prompt(
    base_meter=row["base_meter"],
    form=row["form"],
    num_lines=int(row.get("requested_num_lines", row.get("sft_num_lines"))),
    description=row["enhanced_description"],
)
print(prompt)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
model.eval();

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=220,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
        repetition_penalty=1.05,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

generated = tokenizer.decode(output[0], skip_special_tokens=True)
print(generated)

Notes:

- This notebook uses the intended prompt shape from `description_generation/prompt_contracts.py`.
- The examples come from the released split dataset's `test` split.
- For full training and evaluation details, see `README.md`, `sft/train_sft.py`, and `evaluation/results2.md`.